# SHORE — Interactive Visualisation

This notebook generates a unit-sphere test case from scratch, then inspects
the resulting `.geo` volume grid and `.npz` surface layer using PyVista's HTML
backend, which renders inline as a WebGL canvas — no display or X server required.

**Prerequisites**
```bash
pip install shore[vis]   # adds pyvista
```

In [ ]:
import numpy as np
import pyvista as pv

from shore.io.geo import read_geo, write_geo
from shore.io.vtk import geo_to_pyvista, surface_to_pyvista, write_surface_vts, write_vts
from shore.vis import view_geo, view_surface
from shore.volume.jacobian import jacobian_field, jacobian_per_layer

# Use the HTML (WebGL) backend for inline rendering
pv.set_jupyter_backend('html')

## 0 — Generate a unit-sphere test case

Creates `sphere.stl`, projects it onto a structured `(ni×nj)` surface layer
saved as `sphere_k0.npz`, then marches `nk` wall-normal layers to produce
`sphere.geo`. Skips any step whose output file already exists.

In [ ]:
from pathlib import Path

import trimesh

from shore.surface.io import load_stl
from shore.surface.remesh import project_sphere_to_surface
from shore.volume.hyperbolic import march

NI, NJ, NK = 40, 60, 20
DS, GROWTH = 0.02, 1.10

STL_PATH = Path('sphere.stl')
NPZ_PATH = Path('sphere_k0.npz')
GEO_PATH = Path('sphere.geo')

# ---- Step 1: STL ----
if not STL_PATH.exists():
    m = trimesh.creation.icosphere(subdivisions=4, radius=1.0)
    m.export(str(STL_PATH))
    print(f'Generated {STL_PATH}  ({len(m.faces)} triangles)')
else:
    print(f'{STL_PATH} already exists, skipping.')

# ---- Step 2: surface projection ----
if not NPZ_PATH.exists():
    mesh = load_stl(STL_PATH)
    k0 = project_sphere_to_surface(mesh, ni=NI, nj=NJ)
    np.savez_compressed(str(NPZ_PATH), k0=k0)
    print(f'Generated {NPZ_PATH}  shape={k0.shape}')
else:
    print(f'{NPZ_PATH} already exists, skipping.')
    k0 = np.load(str(NPZ_PATH))['k0']

# ---- Step 3: hyperbolic marching ----
if not GEO_PATH.exists():
    layers_done = []

    def _cb(k: int, jmin: float, step: float) -> None:
        layers_done.append(k)

    grid = march(k0, nk=NK, ds=DS, growth=GROWTH, layer_callback=_cb)
    write_geo(str(GEO_PATH), grid)
    print(f'Generated {GEO_PATH}  shape={grid.shape[:3]}  layers={len(layers_done)}')
else:
    print(f'{GEO_PATH} already exists, skipping.')
    grid = read_geo(GEO_PATH)

## 1 — Load and inspect the volume grid

In [ ]:
grid = read_geo(GEO_PATH)
ni, nj, nk, _ = grid.shape
print(f'Grid dimensions : ni={ni}  nj={nj}  nk={nk}')
print(f'Points          : {ni*nj*nk:,}')
print(f'Cells           : {(ni-1)*nj*(nk-1):,}')

xyz = grid.reshape(-1, 3)
print(f'x bounds : [{xyz[:,0].min():.4f},  {xyz[:,0].max():.4f}]')
print(f'y bounds : [{xyz[:,1].min():.4f},  {xyz[:,1].max():.4f}]')
print(f'z bounds : [{xyz[:,2].min():.4f},  {xyz[:,2].max():.4f}]')

## 2 — Volume grid coloured by Jacobian

Green = high-quality cells, red = poor quality (near inversion).  
The blue surface is the k=0 body wall.

In [ ]:
pl = view_geo(grid, jupyter=True)
pl.show()

## 3 — Per-layer Jacobian profile

Plot the minimum Jacobian as a function of wall-normal layer index.

In [ ]:
import matplotlib.pyplot as plt

jmins = jacobian_per_layer(grid)   # shape (nk-1,)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, len(jmins) + 1), jmins, marker='o', ms=4, lw=1.5)
ax.axhline(0, color='red', lw=0.8, ls='--', label='inversion threshold')
ax.set_xlabel('layer k')
ax.set_ylabel('jmin')
ax.set_title('Minimum cell Jacobian per layer')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Global jmin = {jmins.min():.4e}  at layer {jmins.argmin() + 1}')

## 4 — Jacobian full field on the PyVista mesh

Access the Jacobian array directly from the PyVista object.

In [ ]:
sg = geo_to_pyvista(grid, with_jacobian=True)
print('Cell arrays:', list(sg.cell_data.keys()))
jac = sg.cell_data['Jacobian']
print(f'Jacobian  min={jac.min():.4e}  max={jac.max():.4e}  mean={jac.mean():.4e}')

## 5 — Load and inspect the k=0 surface layer

In [ ]:
k0 = np.load(str(NPZ_PATH))['k0']   # shape (ni, nj, 3)
print(f'Surface shape : {k0.shape}')

radii = np.linalg.norm(k0, axis=-1)
print(f'Radius  mean={radii.mean():.4f}  std={radii.std():.4e}  '
      f'min={radii.min():.4f}  max={radii.max():.4f}')

## 6 — Surface layer coloured by radius from origin

In [ ]:
pl = view_surface(k0, jupyter=True, colorby='radius')
pl.show()

## 7 — Export to VTK for ParaView

Write `.vts` files that ParaView can open directly.

In [ ]:
write_vts('sphere.vts', grid)
write_surface_vts('sphere_k0.vts', k0)

print('Wrote sphere.vts and sphere_k0.vts — open in ParaView.')

## 8 — Slice at a specific k-layer

Inspect an individual wall-normal layer as a structured surface.

In [ ]:
k_slice = nk // 2    # midpoint layer
layer = grid[:, :, k_slice, :]   # shape (ni, nj, 3)

pl = view_surface(layer, jupyter=True, colorby='radius')
pl.add_title(f'Layer k={k_slice}')
pl.show()